In [1]:
import logging
import sys
from pathlib import Path
import pandas as pd
import joblib


logging.basicConfig(
    format="%(asctime)s.%(msecs)d %(levelname)s %(filename)s:%(lineno)d %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger()
logger.setLevel(logging.INFO)


# Move up one level from scripts/ to find the project root
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

logging.info(f"Project root added to path: {PROJECT_ROOT}")

from src.core.dataset_parser import parse_claudette_zipfile
from src.core.feature_engine import SimpleEmbeddingEngine
from src.core.classifier import ClauseClassifier, MODELS_TO_EXPERIMENT




14:29:25.739 INFO 1351323694.py:21 Project root added to path: /home/miguel/enhesa-tos-service
/home/miguel/.cache/pypoetry/virtualenvs/enhesa-tos-service-eiCNO2e5-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
artifacts_exp="simple_embeddings"
#artifacts_exp="legal_bert_embeddings"

zip_filepath = PROJECT_ROOT / "data" / "ToS.zip"
artifacts_dir = PROJECT_ROOT / "models" / artifacts_exp
models_output_dir = artifacts_dir / "classifiers"


In [3]:

rows = []

for model_name in MODELS_TO_EXPERIMENT.keys():
    model_folder_path = models_output_dir / model_name
    
    # Skip if the directory doesn't exist yet
    if not model_folder_path.exists():
        continue
        
    report_file_path = model_folder_path / "evaluation_report.joblib"
    metrics_payload = joblib.load(report_file_path)
    
    # 1. Build a clean dictionary for this row
    row_data = {
        "Model": model_name,
        "Macro F1": metrics_payload.get("macro_f1"),
        "Precision": metrics_payload.get("macro_precision"),
        "Recall": metrics_payload.get("macro_recall"),
        "Accuracy": metrics_payload.get("accuracy"),
        # Stringify the hyperparameters so they fit nicely into a single cell
    }
    
    rows.append(row_data)

# ensemble
model_name="MetaEnsemble_Stacking"
model_folder_path = models_output_dir / model_name

    
report_file_path = model_folder_path / "evaluation_report.joblib"
metrics_payload = joblib.load(report_file_path)

# 1. Build a clean dictionary for this row
row_data = {
    "Model": model_name,
    "Macro F1": metrics_payload.get("macro_f1"),
    "Precision": metrics_payload.get("macro_precision"),
    "Recall": metrics_payload.get("macro_recall"),
    "Accuracy": metrics_payload.get("accuracy"),
    # Stringify the hyperparameters so they fit nicely into a single cell
}

rows.append(row_data)

# 2. Convert the collected rows into a structured Pandas DataFrame
df_results = pd.DataFrame(rows)






In [4]:
print(df_results)

                   Model  Macro F1  Precision    Recall  Accuracy
0     LogisticRegression  0.727416   0.691178  0.842342  0.848115
1              LinearSVC  0.731772   0.694674  0.846260  0.851301
2       SVC_RBF_Pipeline  0.854551   0.864273  0.845508  0.944769
3           RandomForest  0.752610   0.754814  0.750462  0.904408
4   HistGradientBoosting  0.714960   0.855354  0.664238  0.917685
5          MLPClassifier  0.835156   0.874236  0.805397  0.941583
6  MetaEnsemble_Stacking  0.804343   0.757673  0.901963  0.901221
